# Rekurencyjne sieci neuronowe

W poprzednim module omówiliśmy bogate reprezentacje semantyczne tekstu. Architektura, której używaliśmy, uchwytuje zagregowane znaczenie słów w zdaniu, ale nie uwzględnia **kolejności** słów, ponieważ operacja agregacji następująca po embeddingach usuwa tę informację z oryginalnego tekstu. Ponieważ te modele nie są w stanie reprezentować porządku słów, nie potrafią rozwiązać bardziej złożonych lub niejednoznacznych zadań, takich jak generowanie tekstu czy odpowiadanie na pytania.

Aby uchwycić znaczenie sekwencji tekstu, użyjemy architektury sieci neuronowej zwanej **rekurencyjną siecią neuronową**, czyli RNN. Korzystając z RNN, wprowadzamy nasze zdanie do sieci po jednym tokenie na raz, a sieć generuje pewien **stan**, który następnie ponownie przekazujemy do sieci wraz z kolejnym tokenem.

![Obraz pokazujący przykład generowania w rekurencyjnej sieci neuronowej.](../../../../../translated_images/pl/rnn.27f5c29c53d727b5.webp)

Dając sekwencję wejściowych tokenów $X_0,\dots,X_n$, RNN tworzy sekwencję bloków sieci neuronowej i trenuje tę sekwencję end-to-end za pomocą wstecznej propagacji. Każdy blok sieci bierze na wejście parę $(X_i,S_i)$ i zwraca wynik $S_{i+1}$. Końcowy stan $S_n$ lub wyjście $Y_n$ trafia do liniowego klasyfikatora, który produkuje wynik. Wszystkie bloki sieci dzielą te same wagi i są trenowane end-to-end przy użyciu jednej rundy propagacji wstecznej.

> Powyższy rysunek pokazuje rekurencyjną sieć neuronową w rozwiniętej formie (po lewej) oraz w bardziej zwartej reprezentacji rekurencyjnej (po prawej). Ważne jest, aby zrozumieć, że wszystkie komórki RNN mają te same **współdzielone wagi**.

Ponieważ wektory stanu $S_0,\dots,S_n$ są przekazywane przez sieć, RNN jest w stanie nauczyć się zależności sekwencyjnych między słowami. Na przykład, gdy w sekwencji pojawia się słowo *not* (nie), może nauczyć się negować pewne elementy wektora stanu.

Wewnątrz każda komórka RNN zawiera dwie macierze wag: $W_H$ i $W_I$ oraz bias $b$. Na każdym kroku RNN, mając dane wejściowe $X_i$ oraz stan wejściowy $S_i$, stan wyjściowy obliczany jest jako $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, gdzie $f$ to funkcja aktywacji (często $\tanh$).

> W przypadku problemów takich jak generowanie tekstu (które omówimy w następnym rozdziale) lub tłumaczenie maszynowe chcemy też otrzymać wartość wyjściową na każdym kroku RNN. W takim przypadku istnieje dodatkowa macierz $W_O$, a wyjście obliczane jest jako $Y_i=f(W_O\times S_i+b_O)$.

Zobaczmy, jak rekurencyjne sieci neuronowe mogą pomóc nam sklasyfikować nasz zbiór danych wiadomości.

> W środowisku sandbox musimy uruchomić następującą komórkę, aby upewnić się, że wymagana biblioteka jest zainstalowana, a dane wstępnie pobrane. Jeśli uruchamiasz lokalnie, możesz pominąć następną komórkę.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Podczas trenowania dużych modeli alokacja pamięci GPU może stać się problemem. Może także zajść potrzeba eksperymentowania z różnymi rozmiarami minibatchy, tak aby dane mieściły się w pamięci GPU, a jednocześnie trening był na tyle szybki. Jeśli uruchamiasz ten kod na własnym komputerze z GPU, możesz eksperymentować z dostosowaniem rozmiaru minibatcha, aby przyspieszyć trening.

> **Uwaga**: Pewne wersje sterowników NVidia są znane z tego, że nie zwalniają pamięci po wytrenowaniu modelu. Uruchamiamy kilka przykładów w tych notatnikach i może to powodować wyczerpanie pamięci w niektórych konfiguracjach, szczególnie jeśli wykonujesz własne eksperymenty w ramach tego samego notatnika. Jeśli napotkasz dziwne błędy podczas rozpoczynania trenowania modelu, warto zrestartować jądro notatnika.


In [3]:
batch_size = 16
embed_size = 64

## Prosty klasyfikator RNN

W przypadku prostego RNN każda jednostka rekurencyjna to prosta sieć liniowa, która przyjmuje wektor wejściowy oraz wektor stanu i generuje nowy wektor stanu. W Keras można to przedstawić za pomocą warstwy `SimpleRNN`.

Chociaż możemy przekazać zakodowane metodą one-hot tokeny bezpośrednio do warstwy RNN, nie jest to dobry pomysł ze względu na ich wysoką wymiarowość. Dlatego użyjemy warstwy embeddingowej, by zmniejszyć wymiarowość wektorów słów, potem warstwy RNN, a na końcu klasyfikatora `Dense`.

> **Uwaga**: W przypadkach, gdy wymiarowość nie jest aż tak wysoka, na przykład podczas tokenizacji na poziomie znaków, może mieć sens przekazywanie zakodowanych one-hot tokenów bezpośrednio do komórki RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Uwaga:** Tutaj używamy nietrenowanej warstwy osadzającej dla uproszczenia, ale dla lepszych rezultatów możemy użyć wstępnie wytrenowanej warstwy osadzającej za pomocą Word2Vec, jak opisano w poprzedniej jednostce. Dobrym ćwiczeniem dla Ciebie będzie dostosowanie tego kodu do pracy z wstępnie wytrenowanymi osadzeniami.

Teraz wytrenujmy naszą RNN. RNN są generalnie dość trudne do wytrenowania, ponieważ gdy komórki RNN są rozwijane wzdłuż długości sekwencji, liczba warstw biorących udział w propagacji wstecznej jest dość duża. W związku z tym musimy wybrać mniejszą szybkość uczenia i trenować sieć na większym zbiorze danych, aby uzyskać dobre wyniki. Może to potrwać dość długo, więc preferowane jest użycie GPU.

Aby przyspieszyć proces, będziemy trenować model RNN tylko na tytułach wiadomości, pomijając opis. Możesz spróbować trenować z opisem i zobaczyć, czy uda Ci się wytrenować model.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Uwaga**: dokładność prawdopodobnie będzie tutaj niższa, ponieważ trenujemy tylko na tytułach wiadomości.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Teraz, gdy używamy maskowania, możemy trenować model na całym zbiorze danych tytułów i opisów.

> **Uwaga**: Czy zauważyłeś, że używaliśmy wektoryzera wytrenowanego na tytułach wiadomości, a nie na całym tekście artykułu? Potencjalnie może to spowodować, że niektóre tokeny zostaną zignorowane, dlatego lepiej jest ponownie wytrenować wektoryzer. Jednak może to mieć bardzo niewielki wpływ, więc ze względów uproszczenia pozostaniemy przy poprzednim, wcześniej wytrenowanym wektoryzerze.


## LSTM: pamięć długotrwała krótkiej perspektywy

Jednym z głównych problemów RNN są **zanikające gradienty**. RNN mogą być całkiem długie i mogą mieć trudności z propagowaniem gradientów z powrotem do pierwszej warstwy sieci podczas wstecznej propagacji. Kiedy tak się dzieje, sieć nie może uczyć się zależności między odległymi tokenami. Jednym ze sposobów uniknięcia tego problemu jest wprowadzenie **jawnego zarządzania stanem** za pomocą **bramek**. Dwie najpopularniejsze architektury wprowadzające bramki to **long short-term memory** (LSTM) oraz **gated relay unit** (GRU). Tutaj omówimy LSTM.

![Obraz pokazujący przykład komórki pamięci długotrwałej krótkiej perspektywy](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Sieć LSTM jest zorganizowana podobnie do RNN, ale przekazywane są dwa stany z warstwy do warstwy: aktualny stan $c$ oraz ukryty wektor $h$. W każdej jednostce wektor ukryty $h_{t-1}$ jest łączony z wejściem $x_t$ i razem kontrolują, co dzieje się ze stanem $c_t$ oraz wyjściem $h_{t}$ za pomocą **bramek**. Każda bramka ma funkcję aktywacji sigmoidalną (wyjście w zakresie $[0,1]$), którą można potraktować jako maskę bitową mnożoną przez wektor stanu. LSTM posiada następujące bramki (od lewej do prawej na powyższym obrazku):
* **bramka zapominania**, która decyduje, które składniki wektora $c_{t-1}$ należy zapomnieć, a które przekazać dalej.
* **bramka wejściowa**, która decyduje, ile informacji z wektora wejściowego oraz poprzedniego wektora ukrytego powinno zostać włączone do wektora stanu.
* **bramka wyjściowa**, która bierze nowy wektor stanu i decyduje, które jego składniki zostaną użyte do wygenerowania nowego wektora ukrytego $h_t$.

Składniki stanu $c$ można traktować jako flagi, które można przełączać włączone lub wyłączone. Na przykład, gdy natrafiamy w sekwencji na imię *Alice*, zakładamy, że odnosi się ono do kobiety i podnosimy flagę w stanie, która mówi, że w zdaniu mamy rzeczownik rodzaju żeńskiego. Gdy dalej natrafimy na słowa *and Tom*, podniesiemy flagę, która mówi, że mamy rzeczownik w liczbie mnogiej. Manipulując stanem, możemy więc śledzić właściwości gramatyczne zdania.

> **Uwaga**: Oto świetne źródło do zrozumienia działania LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) autorstwa Christophera Olaha.

Choć wewnętrzna budowa komórki LSTM może wydawać się skomplikowana, Keras ukrywa tę implementację w warstwie `LSTM`, więc jedyne co musimy zrobić w powyższym przykładzie, to zastąpić warstwę rekurencyjną:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Uwaga**: trenowanie LSTM-ów również jest dość powolne i na początku treningu możesz nie zauważyć dużego wzrostu dokładności. Może być konieczne kontynuowanie treningu przez pewien czas, aby osiągnąć dobrą dokładność.


## Dwukierunkowe i wielowarstwowe RNN

W naszych dotychczasowych przykładach sieci rekurencyjne działają od początku sekwencji aż do jej końca. Wydaje się to naturalne, ponieważ podąża to za tym samym kierunkiem, w którym czytamy lub słuchamy mowy. Jednakże w scenariuszach, które wymagają losowego dostępu do sekwencji wejściowej, bardziej sensowne jest uruchamianie obliczeń rekurencyjnych w obu kierunkach. RNN, które pozwalają na obliczenia w obu kierunkach, nazywa się **dwukierunkowymi** RNN i można je utworzyć, opakowując warstwę rekurencyjną w specjalną warstwę `Bidirectonal`.

> **Uwaga**: Warstwa `Bidirectional` tworzy dwie kopie warstwy w niej zawartej i ustawia właściwość `go_backwards` jednej z tych kopii na `True`, powodując, że działa ona w przeciwnym kierunku wzdłuż sekwencji.

Sieci rekurencyjne, jednokierunkowe lub dwukierunkowe, wykrywają wzorce w sekwencji i przechowują je w wektorach stanu lub zwracają je jako wyjście. Podobnie jak w sieciach splotowych, możemy zbudować kolejną warstwę rekurencyjną po pierwszej, aby wychwycić wyższe poziomy wzorców, zbudowanych z niższych poziomów wzorców wyodrębnionych przez pierwszą warstwę. Prowadzi to nas do pojęcia **wielowarstwowego RNN**, który składa się z co najmniej dwóch sieci rekurencyjnych, gdzie wyjście poprzedniej warstwy jest przekazywane do następnej warstwy jako wejście.

![Obraz pokazujący wielowarstwowe RNN z długą pamięcią krótkotrwałą](../../../../../translated_images/pl/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Zdjęcie z [tego wspaniałego posta](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) autorstwa Fernando Lópeza.*

Keras ułatwia budowę tych sieci, ponieważ wystarczy dodać więcej warstw rekurencyjnych do modelu. Dla wszystkich warstw z wyjątkiem ostatniej, musimy określić parametr `return_sequences=True`, ponieważ potrzebujemy, aby warstwa zwracała wszystkie stany pośrednie, a nie tylko końcowy stan obliczeń rekurencyjnych.

Zbudujmy dwuwarstwowy dwukierunkowy LSTM dla naszego problemu klasyfikacji.

> **Uwaga** ten kod ponownie zajmuje dość dużo czasu na wykonanie, ale zapewnia najwyższą dokładność, jaką dotychczas widzieliśmy. Może więc warto poczekać i zobaczyć wynik.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN-y dla innych zadań

Do tej pory koncentrowaliśmy się na używaniu RNN do klasyfikacji sekwencji tekstu. Ale potrafią one poradzić sobie z wieloma innymi zadaniami, takimi jak generowanie tekstu i tłumaczenie maszynowe &mdash; rozważymy te zadania w następnej jednostce.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
